In [1]:
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import pickle
import random
import matplotlib.pyplot as plt

from joblib import dump, load
from os import path
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from matplotlib.pyplot import *
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)

In [ ]:
# Load the model
# with open('random_forest_model.pkl', 'rb') as f:
#     model = pickle.load(f)

In [2]:
# Read the CSV file into a DataFrame
path = r'C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV'
all_files = glob.glob(path + "/*.csv")
li = []
for i in range(10):
    df = pd.read_csv(all_files[i], encoding='cp1252', index_col=None, header=0)
    li.append(df)
    print("Read Completed for ", all_files[i])
df = pd.concat(li, axis=0, ignore_index=True)
df.describe()
df.head()
df.info()

Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged01.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged02.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged03.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged04.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged05.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged06.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged07.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV\Merged08.csv
Read Completed for  C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-202

In [3]:
delete_cat = ['DDOS-RSTFINFLOOD', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD', 'DDOS-TCP_FLOOD', 'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'RECON-OSSCAN', 'RECON-PORTSCAN']

df = df[~df['Label'].isin(delete_cat)].reset_index(drop=True)

In [4]:
# Grouping
DDoS = ['DDOS-ACK_FRAGMENTATION',     
        'DDOS-UDP_FLOOD',       
        'DDOS-SLOWLORIS',               
        'DDOS-ICMP_FLOOD',        
        'DDOS-RSTFINFLOOD',    
        'DDOS-PSHACK_FLOOD',        
        'DDOS-HTTP_FLOOD',              
        'DDOS-UDP_FRAGMENTATION',     
        'DDOS-TCP_FLOOD',  
        'DDOS-SYN_FLOOD',        
        'DDOS-SYNONYMOUSIP_FLOOD',    
        'DDOS-ICMP_FRAGMENTATION']

DoS = ['DOS-UDP_FLOOD',    
       'DOS-TCP_FLOOD',    
       'DOS-SYN_FLOOD',    
       'DOS-HTTP_FLOOD']

Spoofing = ['MITM-ARPSPOOFING',     
            'DNS_SPOOFING']

Brute_force = ['DICTIONARYBRUTEFORCE']

Recon = ['RECON-HOSTDISCOVERY', 
         'RECON-OSSCAN',     
         'RECON-PORTSCAN',               
         'RECON-PINGSWEEP',         
         'VULNERABILITYSCAN']

Web_based = ['SQLINJECTION',        
             'BROWSERHIJACKING',         
             'COMMANDINJECTION',       
             'XSS',         
             'BACKDOOR_MALWARE',          
             'UPLOADING_ATTACK']

Mirai = ['MIRAI-GREETH_FLOOD',     
         'MIRAI-UDPPLAIN',     
         'MIRAI-GREIP_FLOOD']



def classify_attacks(data):
    data['Label'].replace(DDoS,'DDoS',inplace=True)
    data['Label'].replace(DoS,'DoS',inplace=True)
    data['Label'].replace(Spoofing,'Spoofing',inplace=True)
    data['Label'].replace(Brute_force,'Brute_Force',inplace=True)
    data['Label'].replace(Recon,'Recon',inplace=True)
    data['Label'].replace(Web_based,'Web_based',inplace=True)
    data['Label'].replace(Mirai,'Mirai',inplace=True)
classify_attacks(df)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_11152\387110630.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Label'].replace(DDoS,'DDoS',inplace=True)


In [4]:
# Train set preprocessing
df.drop_duplicates(keep='first', inplace = True)
df.replace([np.inf, -np.inf], np.nan, inplace=True) # Replace inf and -inf with NaN, then drop the resulting NaNs
df.dropna(inplace=True)    # dropna() removes any rows that contain NaN (missing) values; reset_index() resets the row index after dropping, so it's clean and continuous.
df.reset_index(drop=True, inplace=True)

# data_clean = df.drop(columns=['fin_count', 'fin_flag_number', 'rst_count', 'rst_flag_number', 'Tot size', 'syn_count', 'ack_count', 'SMTP', 'Telnet', 'IRC', 'cwr_flag_number', 'IGMP', 'Number', 'LLC', 'ece_flag_number', 'DHCP', 'SSH', 'Protocol Type', 'IPv'])
data_clean = df.drop(columns=['fin_count', 'fin_flag_number', 'rst_count', 'rst_flag_number', 'syn_count', 'syn_flag_number', 'Number'])

In [29]:
columns_name = data_clean.drop(columns=['Label']).columns

In [30]:
# X_test = df

In [5]:
# Create test data
X_train_val = data_clean.drop(columns=['Label'])
Y_train_val = data_clean['Label']

In [6]:
test = pd.read_csv('C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV/Merged11.csv')

test_delete_cat = ['DDOS-RSTFINFLOOD', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD', 'DDOS-TCP_FLOOD', 'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'RECON-OSSCAN', 'RECON-PORTSCAN']
test = test[~test['Label'].isin(test_delete_cat)].reset_index(drop=True)

# Test set preprocessing
test.drop_duplicates(keep='first', inplace = True)
test.replace([np.inf, -np.inf], np.nan, inplace=True) # Replace inf and -inf with NaN, then drop the resulting NaNs
test.dropna(inplace=True)    # dropna() removes any rows that contain NaN (missing) values; reset_index() resets the row index after dropping, so it's clean and continuous.
test.reset_index(drop=True, inplace=True)

# data_clean = test.drop(columns=['fin_count', 'fin_flag_number', 'rst_count', 'rst_flag_number', 'Tot size', 'syn_count', 'ack_count', 'SMTP', 'Telnet', 'IRC', 'cwr_flag_number', 'IGMP', 'Number', 'LLC', 'ece_flag_number', 'DHCP', 'SSH', 'Protocol Type', 'IPv'])
test = test.drop(columns=['fin_count', 'fin_flag_number', 'rst_count', 'rst_flag_number', 'syn_count', 'syn_flag_number', 'Number'])

X_test = test.drop(columns=['Label'])
Y_test = test['Label']

In [17]:
print(X_test)

        Header_Length  Protocol Type  Time_To_Live          Rate  \
0               20.00              6         64.00   6594.921304   
1                8.00             17         64.00  11506.059858   
2               20.00              6         64.00  25503.490210   
3               32.00              6        164.80    162.770547   
4                0.00              1         64.00  64887.128713   
...               ...            ...           ...           ...   
548903           0.08              1         62.72   2095.904937   
548904           8.00             17         64.00  34022.582738   
548905          20.00              6         64.00  11551.056154   
548906          20.00              6         64.00  15197.854917   
548907          19.80              6         63.36   1808.217005   

        psh_flag_number  ack_flag_number  ece_flag_number  cwr_flag_number  \
0                   1.0              1.0              0.0              0.0   
1                   0.0    

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_val)
X_test_scaled = scaler.fit_transform(X_test)

In [8]:
# Split the data set into training and testing
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_scaled, Y_train_val, test_size=0.25, random_state=33, shuffle=True)

In [9]:
frst_model = RandomForestClassifier(n_estimators=100,       # Default (reduce to 50-80 if too slow)
                                    max_depth=20,           # Limit tree depth to avoid overfitting + speed up
                                    min_samples_split=50,   # Reduce splits on small nodes
                                    min_samples_leaf=20,    # Prevent tiny leaves
                                    max_features='sqrt',    # Faster: only sqrt(features) per split
                                    n_jobs=-1,              # Use all CPU cores
                                    verbose=1,              # Show progress
                                    random_state=42)        # Reproducibility

frst_model.fit(X_train, Y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.0min finished


RandomForestClassifier(max_depth=20, min_samples_leaf=20, min_samples_split=50,
                       n_jobs=-1, random_state=42, verbose=1)

In [10]:
y_pred_frst = frst_model.predict(X_val)
print(classification_report(Y_val, y_pred_frst))

[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    2.5s finished
c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                         precision    recall  f1-score   support

       BACKDOOR_MALWARE       0.00      0.00      0.00       110
                 BENIGN       0.80      0.93      0.86     42592
       BROWSERHIJACKING       1.00      0.05      0.10       231
       COMMANDINJECTION       1.00      0.13      0.23       217
 DDOS-ACK_FRAGMENTATION       1.00      0.98      0.99     11200
        DDOS-HTTP_FLOOD       0.87      0.66      0.75      1085
        DDOS-ICMP_FLOOD       1.00      1.00      1.00    106802
DDOS-ICMP_FRAGMENTATION       0.99      0.98      0.99     17388
      DDOS-PSHACK_FLOOD       1.00      1.00      1.00     85713
         DDOS-SLOWLORIS       0.67      0.86      0.75       929
         DDOS-UDP_FLOOD       0.71      0.87      0.78     99788
 DDOS-UDP_FRAGMENTATION       0.99      0.99      0.99     11053
   DICTIONARYBRUTEFORCE       0.96      0.12      0.21       539
           DNS_SPOOFING       0.79      0.58      0.67      6935
         DOS-HTTP_FLOOD 

c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
test_pred = frst_model.predict(X_test_scaled)
print(classification_report(Y_test, test_pred))

[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    1.4s finished
c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                         precision    recall  f1-score   support

       BACKDOOR_MALWARE       0.00      0.00      0.00        58
                 BENIGN       0.72      0.90      0.80     21196
       BROWSERHIJACKING       1.00      0.02      0.03       130
       COMMANDINJECTION       0.75      0.07      0.13        83
 DDOS-ACK_FRAGMENTATION       1.00      0.99      0.99      5507
        DDOS-HTTP_FLOOD       0.74      0.61      0.66       578
        DDOS-ICMP_FLOOD       1.00      1.00      1.00     81101
DDOS-ICMP_FRAGMENTATION       1.00      0.98      0.99      8842
      DDOS-PSHACK_FLOOD       1.00      0.99      1.00     59118
         DDOS-SLOWLORIS       0.46      0.82      0.59       458
         DDOS-UDP_FLOOD       0.62      0.00      0.00     70735
 DDOS-UDP_FRAGMENTATION       0.99      0.99      0.99      5581
   DICTIONARYBRUTEFORCE       1.00      0.08      0.15       259
           DNS_SPOOFING       0.70      0.38      0.49      3424
         DOS-HTTP_FLOOD 

c:\Users\ADMIN\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
# for i in range(50):
#     # sample = X_test.iloc[[i]]  # Must be 2D
#     sample = X_scaled[i+50].reshape(1, -1)  #If standard scaler is includeed, always reshape the data
#     pred = model.predict(sample)[0]

#     print(f"Sample {i}: Predicted = {pred}")

In [34]:
for i in range(50):
    # sample = X_test.iloc[[i]]  # Must be 2D
    sample = X_scaled[i+50].reshape(1, -1)  #If standard scaler is includeed, always reshape the data
    pred = model.predict(sample)[0]
    actual = Y_test.iloc[i]

    print(f"Sample {i}: Predicted = {pred}, Actual = {actual}")

# rand = random.randint(0,len(X_scaled)-1)
# test_data = X_scaled[rand].reshape(1, -1)    #If standard scaler is includeed, always reshape the data

# pred_label = model.predict(test_data)
# print("Predicted class label:", pred_label)
# real_label = Y_test.iloc[rand]
# print("Real class label:", real_label)

[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   3 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   4 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   5 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   6 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   7 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   8 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  11 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  12 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  13 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  14 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  15 tasks      | elaps

In [ ]:
# Check feature importance
feat_importances = pd.Series(model.feature_importances_, index=columns_name)
top_features = feat_importances.sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=top_features, y=top_features.index)
plt.title("Top 20 Feature Importances in Random Forest")
plt.show()